# Fase 2 — NER com BERTimbau

Fine-tuning do **BERTimbau** (`neuralmind/bert-base-portuguese-cased`) para reconhecer as entidades
`EXAME`, `VALOR`, `UNIDADE` e `REFERENCIA` em laudos de texto livre (Token Classification, esquema BIO).

A avaliação compara com a linha de base de regex nos splits:
- **teste**: formatos de laudo vistos no treino;
- **teste_ood**: formatos **nunca vistos** (tabela com a referência antes do valor e laudo em várias linhas).

> No Colab: *Ambiente de execução → Alterar o tipo → GPU T4*. Também roda na CPU (~30–40 min).

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    # Repositório privado? Use https://<usuario>:<token>@gitlab.com/... ou envie um .zip do projeto.
    if not os.path.exists("caresync-nlp-core"):
        !git clone -q https://gitlab.com/caiofasaraiva/caresync-nlp-core.git
    %cd caresync-nlp-core
    !pip install -q "transformers>=5" datasets accelerate sentencepiece protobuf seqeval pydantic-settings groq fastapi
else:
    %cd ..
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (CPU)")

## Dataset
Gerado por `scripts/gerar_dataset.py` (seed fixa, reprodutível).

In [ ]:
!python scripts/gerar_dataset.py --n 1200 --seed 42

## Linha de base: regex + regras

In [ ]:
!python scripts/avaliar_regex.py

## Treinamento do NER

In [ ]:
!python scripts/treinar_ner.py --epocas 4 --lr 5e-5 --batch 8

## Curva de loss

In [ ]:
ARQUIVO = "resultados/ner_ner-bertimbau.json"
TITULO = "BERTimbau NER"

import json
import matplotlib.pyplot as plt

res = json.load(open(ARQUIVO, encoding="utf-8"))
hist = res["historico"]
treino = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
valid = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*treino), label="treino")
plt.plot(*zip(*valid), "o-", label="validação")
plt.xlabel("época"); plt.ylabel("loss"); plt.title(TITULO); plt.legend(); plt.grid(alpha=.3)
plt.savefig(ARQUIVO.replace(".json", "_loss.png"), dpi=200, bbox_inches="tight")
plt.show()

## Comparação: regex x NER (pipeline completo até o nível de risco)

In [ ]:
import pandas as pd

base = json.load(open("resultados/baseline_regex.json", encoding="utf-8"))
linhas = []
for split in ("teste", "teste_ood"):
    for nome, ext, tri in [
        ("Regex + regras", base[split]["extracao"], base[split]["triagem"]),
        ("NER + regras", res[split]["pipeline_ner_regras"]["extracao"], res[split]["pipeline_ner_regras"]["triagem"]),
    ]:
        linhas.append({"split": split, "sistema": nome, "F1 extração": ext["f1"], "recall extração": ext["recall"],
                       "acurácia risco": tri["acuracia"], "recall CRÍTICO": tri["por_classe"]["CRITICO"]["recall"],
                       "críticos perdidos": tri["criticos_perdidos"]})
pd.DataFrame(linhas).round(3)

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split in zip(axes, ("teste", "teste_ood")):
    m = np.array(res[split]["pipeline_ner_regras"]["triagem"]["matriz_confusao"])
    ax.imshow(m, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, m[i, j], ha="center", va="center")
    ax.set_xticks(range(3), ["NORMAL", "ATENÇÃO", "CRÍTICO"]); ax.set_yticks(range(3), ["NORMAL", "ATENÇÃO", "CRÍTICO"])
    ax.set_xlabel("previsto"); ax.set_ylabel("real"); ax.set_title(f"NER + regras — {split}")
plt.tight_layout(); plt.savefig("resultados/ner_matriz_confusao.png", dpi=200); plt.show()

## Baixar o modelo treinado
Coloque a pasta em `modelos/ner-bertimbau` e use `EXTRATOR=ner` na API.

In [ ]:
if "google.colab" in sys.modules:
    !zip -qr ner-bertimbau.zip modelos/ner-bertimbau -x "modelos/ner-bertimbau/checkpoints/*"
    from google.colab import files
    files.download("ner-bertimbau.zip")